# CodeAlpha Machine Learning Internship
## Task 1: Credit Scoring Model

### 🎯 Objective
Predict an individual's creditworthiness (`Good`, `Standard`, `Poor`) using past financial history, income, debt levels, payment behaviors, and credit utilization.

### 🛠️ Key Components
- **Dataset**: Kaggle Credit Score Classification (`parisrohan/credit-score-classification` via `kagglehub`)
- **Feature Engineering**: Debt-to-Income, EMI-to-Salary, Delay Severity, Credit History Months
- **Algorithms Evaluated**: Logistic Regression, Decision Tree, Random Forest, HistGradientBoosting
- **Evaluation Metrics**: Accuracy, Precision, Recall, F1-Score (Macro & Weighted), ROC-AUC

### Step 1: Environment Setup & Library Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for modular imports
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import clean_dataframe, prepare_train_test_data
from src.evaluate import calculate_metrics, plot_confusion_matrix, plot_multiclass_roc

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

### Step 2: Load the Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "train.csv"

if not data_path.exists():
    print("Dataset not found locally. Running download...")
    from src.download_data import download_from_kagglehub
    download_from_kagglehub()

df_raw = pd.read_csv(data_path)
print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

### Step 3: Exploratory Data Analysis (EDA)
Let's explore class distributions, income levels, and correlations.

In [ ]:
# Target distribution
plt.figure(figsize=(7, 4))
palette = {"Good": "#2ecc71", "Standard": "#f39c12", "Poor": "#e74c3c"}
sns.countplot(data=df_raw, x="Credit_Score", palette=palette, order=["Good", "Standard", "Poor"])
plt.title("Target Distribution: Credit Score Categories", fontsize=13)
plt.xlabel("Credit Score")
plt.ylabel("Applicant Count")
plt.show()

### Step 4: Data Preprocessing & Feature Engineering
We clean raw artifacts (e.g. trailing underscores in numeric columns), parse credit history strings into total months, and compute key financial ratios:
1. `Debt_to_Income_Ratio`
2. `EMI_to_Salary_Ratio`
3. `Delay_Severity` (`Delay_from_due_date` × `Num_of_Delayed_Payment`)
4. `Savings_to_Salary_Ratio`

In [ ]:
X_train, X_test, y_train, y_test, preprocessor = prepare_train_test_data(data_path)
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Total features engineered & encoded: {len(preprocessor.feature_names)}")

### Step 5: Train and Benchmark Candidate Models
We evaluate four models:
1. **Logistic Regression** (Baseline)
2. **Decision Tree Classifier**
3. **Random Forest Classifier**
4. **Histogram Gradient Boosting Classifier**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=14, random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=42)
}

results = []
fitted_models = {}

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train, y_train)
    fitted_models[name] = clf
    metrics = calculate_metrics(clf, X_test, y_test)
    results.append({
        "Model": name,
        "Accuracy": round(metrics["Accuracy"], 4),
        "Precision (Macro)": round(metrics["Precision (Macro)"], 4),
        "Recall (Macro)": round(metrics["Recall (Macro)"], 4),
        "F1-Score (Macro)": round(metrics["F1-Score (Macro)"], 4),
        "ROC-AUC (Macro)": round(metrics["ROC-AUC (Macro)"], 4) if metrics["ROC-AUC (Macro)"] else 0.0
    })

benchmark_df = pd.DataFrame(results).sort_values(by="F1-Score (Macro)", ascending=False)
benchmark_df

### Step 6: Visual Model Benchmark Comparison

In [ ]:
plt.figure(figsize=(10, 5))
benchmark_melted = benchmark_df.melt(id_vars="Model", value_vars=["Accuracy", "F1-Score (Macro)", "ROC-AUC (Macro)"])
sns.barplot(data=benchmark_melted, x="Model", y="value", hue="variable", palette="magma")
plt.title("Credit Score Classification: Model Benchmark Performance", fontsize=13)
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.show()

### Step 7: Best Model Detailed Evaluation & Confusion Matrix

In [ ]:
best_name = benchmark_df.iloc[0]["Model"]
best_clf = fitted_models[best_name]
print(f"Best performing algorithm: {best_name}\n")

best_metrics = calculate_metrics(best_clf, X_test, y_test)
print(best_metrics["Classification Report"])

# Normalized Confusion Matrix Heatmap
cm = best_metrics["Confusion Matrix"]
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=["Poor", "Standard", "Good"], yticklabels=["Poor", "Standard", "Good"])
plt.title(f"Confusion Matrix: {best_name}", fontsize=13)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

### Step 8: Multi-Class ROC-AUC Curves

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_prob = best_clf.predict_proba(X_test)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
target_labels = ["Poor", "Standard", "Good"]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]

plt.figure(figsize=(8, 6))
for i, color in enumerate(colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{target_labels[i]} (AUC = {roc_score:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curves (One-vs-Rest) - {best_name}", fontsize=13)
plt.legend(loc="lower right")
plt.show()

### Step 9: Feature Importance Analysis

In [ ]:
rf_model = fitted_models["Random Forest"]
importances = rf_model.feature_importances_
top_indices = np.argsort(importances)[::-1][:15]
top_feature_names = [preprocessor.feature_names[i] for i in top_indices]
top_scores = [importances[i] for i in top_indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_scores, y=top_feature_names, palette="viridis")
plt.title("Top 15 Most Predictive Financial Features (Random Forest)", fontsize=13)
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.show()

### Step 10: Conclusion & Internship Highlights

1. **Model Performance**: Tree-based ensembles (Random Forest / Gradient Boosting) significantly outperform linear baselines on creditworthiness classification.
2. **Critical Features**: Outstanding Debt, Payment Delay Days, Delay Severity ratio, and Credit History length are the primary drivers of an individual's credit score.
3. **Deliverables Ready**:
   - Pretrained model pipeline exported to `models/`
   - Standalone CLI inference module in `src/predict.py`
   - Interactive Streamlit Dashboard in `visualizer/app.py`